# **Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# **Data Reading**


In [2]:
data = pd.read_csv("Food_Delivery_Time_Prediction.csv")
print(data.shape)
data.head()


(50000, 24)


,Order_ID,Order_Date,Order_Hour,Day_of_Week,Is_Weekend,Is_Festival,Weather,Pickup_Zone,Dropoff_Zone,Vehicle_Type,...,Order_Items,Restaurant_Load,Preparation_Time_Min,Road_Distance_km,Delivery_Distance_Category,Traffic_Level,Number_of_Signals,Average_Speed_kmph,Delivery_Priority,Time_taken_min
0,ORD000001,2025-10-28,8,Tuesday,0,0,Clear,CBD,Commercial,Scooter,...,1,Medium,23,30.55,Long,Moderate,15,28.7,Normal,91
1,ORD000002,2025-09-26,12,Friday,0,0,Rain,CBD,CBD,Scooter,...,3,Medium,11,2.25,Short,Moderate,17,23.3,Normal,21
2,ORD000003,2025-01-26,18,Sunday,1,0,Clear,Residential,Residential,Scooter,...,1,Low,31,10.15,Long,Moderate,8,28.9,Normal,52
3,ORD000004,2025-11-19,12,Wednesday,0,0,Cloudy,Residential,Industrial,Bike,...,3,Medium,19,26.33,Long,Low,9,41.4,Normal,59
4,ORD000005,2025-08-04,18,Monday,0,0,Rain,Industrial,Residential,Bike,...,1,Medium,7,30.48,Long,Moderate,8,31.7,Normal,75


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Order_ID                    50000 non-null  str    
 1   Order_Date                  50000 non-null  str    
 2   Order_Hour                  50000 non-null  int64  
 3   Day_of_Week                 50000 non-null  str    
 4   Is_Weekend                  50000 non-null  int64  
 5   Is_Festival                 50000 non-null  int64  
 6   Weather                     50000 non-null  str    
 7   Pickup_Zone                 50000 non-null  str    
 8   Dropoff_Zone                50000 non-null  str    
 9   Vehicle_Type                50000 non-null  str    
 10  Rider_Experience_Years      50000 non-null  float64
 11  Rider_Rating                50000 non-null  float64
 12  Restaurant_Rating           50000 non-null  float64
 13  Cuisine_Type                50000 non-null

In [4]:
data.isnull().sum()

Order_ID                      0
Order_Date                    0
Order_Hour                    0
Day_of_Week                   0
Is_Weekend                    0
Is_Festival                   0
Weather                       0
Pickup_Zone                   0
Dropoff_Zone                  0
Vehicle_Type                  0
Rider_Experience_Years        0
Rider_Rating                  0
Restaurant_Rating             0
Cuisine_Type                  0
Order_Items                   0
Restaurant_Load               0
Preparation_Time_Min          0
Road_Distance_km              0
Delivery_Distance_Category    0
Traffic_Level                 0
Number_of_Signals             0
Average_Speed_kmph            0
Delivery_Priority             0
Time_taken_min                0
dtype: int64

In [5]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
Order_Hour,50000.0,13.847700,5.525291,0.00,10.00,14.00,19.00,23.00
Is_Weekend,50000.0,0.283600,0.450749,0.00,0.00,0.00,1.00,1.00
Is_Festival,50000.0,0.049100,0.216079,0.00,0.00,0.00,0.00,1.00
Rider_Experience_Years,50000.0,4.439744,2.927752,0.10,2.20,3.80,5.90,15.00
Rider_Rating,50000.0,3.951646,0.294960,3.20,3.70,3.90,4.10,5.00
Restaurant_Rating,50000.0,4.175368,0.356201,3.30,3.90,4.20,4.40,5.00
Order_Items,50000.0,2.652740,1.590359,1.00,1.00,2.00,4.00,7.00
Preparation_Time_Min,50000.0,24.031180,8.348770,5.00,18.00,24.00,30.00,51.00
Road_Distance_km,50000.0,26.201744,12.620079,0.08,16.33,25.62,35.48,70.82
Number_of_Signals,50000.0,9.728920,4.330849,2.00,6.00,9.00,14.00,18.00


In [6]:
data.describe(exclude=np.number).T

,count,unique,top,freq
Order_ID,50000,50000,ORD000001,1
Order_Date,50000,365,2025-02-12,170
Day_of_Week,50000,7,Tuesday,7264
Weather,50000,5,Clear,23992
Pickup_Zone,50000,5,Residential,15786
Dropoff_Zone,50000,5,Residential,16106
Vehicle_Type,50000,4,Bike,22943
Cuisine_Type,50000,9,North Indian,9483
Restaurant_Load,50000,3,Low,17751
Delivery_Distance_Category,50000,3,Long,47238


### **Observations**:
1. No null values are present in any column.

2. Both numerical features *(Rider_Experience_Years, Road_Distance_km, Preparation_Time_Min, ...)* and categorical features *(Weather, Traffic_Level, Vehicle_Type, ...)* will require encoding before model training.

3. Order_ID  will not be used to build the model.

4. Weather is clean text, only needs one-hot encoding.

5. Average_Speed_kmph poses a potential data leakage risk as it might be back-calculated from distance and delivery time.

# **Exploratory Data Analysis**


In [7]:
fig = px.box(data, x="Weather", y="Time_taken_min", title="Estimated delivery time based on weather conditions")
fig.show()


In [8]:
fig = px.scatter(data, x="Rider_Experience_Years", y="Time_taken_min",
                  trendline="ols", title="Relationship between Driver Experience and Delivery Time")
fig.show()


In [9]:
implied_time = data["Road_Distance_km"] / data["Average_Speed_kmph"] * 60
print("Correlation between calculated duration and actual duration:",
      round(np.corrcoef(implied_time, data["Time_taken_min"])[0, 1], 3))

Correlation between calculated duration and actual duration: 0.933


### **Observations**:

1. **Weather**: Bad weather significantly increases delivery time compared to Clear $\rightarrow$ keep this feature.


2. **Traffic_Level**: Heavy traffic strongly increases delivery duration $\rightarrow$ keep this feature.


3. **Rider_Experience_Years**: Weak non-linear correlation with delivery time ($r \approx -0.06$) $\rightarrow$ tree-based models ( Random Forest) will capture this better than Linear Regression.


4. **Average_Speed_kmph**: High correlation with target, indicating severe **data leakage** $\rightarrow$ drop from feature set.


5. **Road_Distance_km**: Strong positive correlation with delivery time .


# **Data Cleaning**


### **Observations**:
1. **Order_ID:** just an identifier, has no predictive value.
2. **Order_Date:**  the useful time information has already been extracted into
3. **Delivery_Distance_Category:** keeping it would be redundant (we already have the actual km value).
4. **Average_Speed_kmph - DATA LEAKAGE:** computed backwards from Time_taken_min, so it must be removed;

In [10]:
drop_cols = ["Order_ID", "Order_Date", "Delivery_Distance_Category", "Average_Speed_kmph"]
data_model = data.drop(columns=drop_cols)
data_model.head()

,Order_Hour,Day_of_Week,Is_Weekend,Is_Festival,Weather,Pickup_Zone,Dropoff_Zone,Vehicle_Type,Rider_Experience_Years,Rider_Rating,Restaurant_Rating,Cuisine_Type,Order_Items,Restaurant_Load,Preparation_Time_Min,Road_Distance_km,Traffic_Level,Number_of_Signals,Delivery_Priority,Time_taken_min
0,8,Tuesday,0,0,Clear,CBD,Commercial,Scooter,3.1,3.8,3.8,Pizza,1,Medium,23,30.55,Moderate,15,Normal,91
1,12,Friday,0,0,Rain,CBD,CBD,Scooter,1.8,3.6,3.9,Desserts,3,Medium,11,2.25,Moderate,17,Normal,21
2,18,Sunday,1,0,Clear,Residential,Residential,Scooter,3.8,3.9,4.0,Biryani,1,Low,31,10.15,Moderate,8,Normal,52
3,12,Wednesday,0,0,Cloudy,Residential,Industrial,Bike,1.7,3.9,4.6,Burger,3,Medium,19,26.33,Low,9,Normal,59
4,18,Monday,0,0,Rain,Industrial,Residential,Bike,12.4,4.4,4.0,Desserts,1,Medium,7,30.48,Moderate,8,Normal,75


In [11]:
target = "Time_taken_min"

y = data_model[target]
X = data_model.drop(columns=[target])

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)


Categorical columns: ['Day_of_Week', 'Weather', 'Pickup_Zone', 'Dropoff_Zone', 'Vehicle_Type', 'Cuisine_Type', 'Restaurant_Load', 'Traffic_Level', 'Delivery_Priority']
Numerical columns: ['Order_Hour', 'Is_Weekend', 'Is_Festival', 'Rider_Experience_Years', 'Rider_Rating', 'Restaurant_Rating', 'Order_Items', 'Preparation_Time_Min', 'Road_Distance_km', 'Number_of_Signals']


C:\Users\win\AppData\Local\Temp\ipykernel_16032\2972801214.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()



# **Feature Engineering**


In [12]:
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("Number of features post-encoding:", X_encoded.shape[1])
X_encoded.head()


Number of features post-encoding: 46


,Order_Hour,Is_Weekend,Is_Festival,Rider_Experience_Years,Rider_Rating,Restaurant_Rating,Order_Items,Preparation_Time_Min,Road_Distance_km,Number_of_Signals,...,Cuisine_Type_North Indian,Cuisine_Type_Pizza,Cuisine_Type_South Indian,Restaurant_Load_Low,Restaurant_Load_Medium,Traffic_Level_Low,Traffic_Level_Moderate,Traffic_Level_Severe,Delivery_Priority_Priority,Delivery_Priority_VIP
0,8,0,0,3.1,3.8,3.8,1,23,30.55,15,...,False,True,False,False,True,False,True,False,False,False
1,12,0,0,1.8,3.6,3.9,3,11,2.25,17,...,False,False,False,False,True,False,True,False,False,False
2,18,1,0,3.8,3.9,4.0,1,31,10.15,8,...,False,False,False,True,False,False,True,False,False,False
3,12,0,0,1.7,3.9,4.6,3,19,26.33,9,...,False,False,False,False,True,True,False,False,False,False
4,18,0,0,12.4,4.4,4.0,1,7,30.48,8,...,False,False,False,False,True,False,True,False,False,False


In [13]:
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)
print("Train:", Xtrain.shape, "| Test:", Xtest.shape)


Train: (40000, 46) | Test: (10000, 46)


In [14]:
lr_model = LinearRegression()
lr_model.fit(Xtrain, ytrain)
y_pred_lr = lr_model.predict(Xtest)

mae_lr = mean_absolute_error(ytest, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(ytest, y_pred_lr))
r2_lr = r2_score(ytest, y_pred_lr)

print(f"[Linear Regression] MAE: {mae_lr:.2f} minutes | RMSE: {rmse_lr:.2f} minutes | R2: {r2_lr:.3f}")


[Linear Regression] MAE: 6.42 minutes | RMSE: 9.25 minutes | R2: 0.933


# **Model Building**
### **Steps**:
1. Build two regression models — **Linear Regression (baseline) and Random Forest (main model)** — on the training data.
2. Compare the performance of both models and select the better one based on the evaluation results.
3. Evaluate the selected model on the testing data (MAE, RMSE, R²) to assess its accuracy and reliability.






### **Model 1: Linear Regression**






In [15]:
lr_model = LinearRegression()
lr_model.fit(Xtrain, ytrain)
y_pred_lr = lr_model.predict(Xtest)

mae_lr = mean_absolute_error(ytest, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(ytest, y_pred_lr))
r2_lr = r2_score(ytest, y_pred_lr)

print(f"[Linear Regression] MAE: {mae_lr:.2f} minutes | RMSE: {rmse_lr:.2f} minutes | R2: {r2_lr:.3f}")


[Linear Regression] MAE: 6.42 minutes | RMSE: 9.25 minutes | R2: 0.933



### **Model 2: Random Forest Regressor**






In [16]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(Xtrain, ytrain)
y_pred_rf = rf_model.predict(Xtest)

mae_rf = mean_absolute_error(ytest, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(ytest, y_pred_rf))
r2_rf = r2_score(ytest, y_pred_rf)

print(f"[Random Forest] MAE: {mae_rf:.2f} minutes | RMSE: {rmse_rf:.2f} minutes | R2: {r2_rf:.3f}")

[Random Forest] MAE: 3.07 minutes | RMSE: 4.11 minutes | R2: 0.987


In [17]:
importance = pd.Series(rf_model.feature_importances_, index=X_encoded.columns)
importance = importance.sort_values(ascending=False).head(15)


fig = px.bar(importance[::-1], orientation="h", title="Top 15 Features Influencing Delivery Time", labels={"value": "Importance Score", "index": "Feature"},)
fig.show()


# **Test Prediction**


In [18]:
def predict_delivery_time(sample_dict):
    """sample_dict: dict với đầy đủ các cột gốc (trừ target và các cột đã loại bỏ)"""
    sample_df = pd.DataFrame([sample_dict])
    sample_encoded = pd.get_dummies(sample_df)
    sample_encoded = sample_encoded.reindex(columns=X_encoded.columns, fill_value=0)
    return rf_model.predict(sample_encoded)[0]


example_order = {
    "Order_Hour": 8,
    "Day_of_Week": "Tuesday",
    "Is_Weekend": 0,
    "Is_Festival": 0,
    "Weather": "Cloudy",
    "Pickup_Zone": "CBD",
    "Dropoff_Zone": "Commercial",
    "Vehicle_Type": "Scooter",
    "Rider_Experience_Years": 2.8,
    "Rider_Rating": 3.9,
    "Restaurant_Rating": 4.9,
    "Cuisine_Type": "Pizza",
    "Order_Items": 1,
    "Restaurant_Load": "Low",
    "Preparation_Time_Min": 16,
    "Road_Distance_km": 10,
    "Traffic_Level": "Moderate",
    "Number_of_Signals": 14,
    "Delivery_Priority": "Normal",
}

pred_minutes = predict_delivery_time(example_order)

def format_minutes(x):
    h = int(x // 60)
    m = int(x % 60)
    return f"{h} hour(s) {m} minutes" if h > 0 else f"{m} minutes"

print(f"Predicted delivery time: {format_minutes(pred_minutes)}")



Predicted delivery time: 42 minutes



# **Model Deployment**

In [19]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("model_columns.pkl", "wb") as f:
    pickle.dump(list(X_encoded.columns), f)

In [20]:

import pickle
import pandas as pd

with open("model.pkl", "rb") as f:
    rf_model = pickle.load(f)


with open("model_columns.pkl", "rb") as f:
    model_columns = pickle.load(f)
new_order = {
    "Order_Hour": 19, "Day_of_Week": "Saturday", "Is_Weekend": 1, "Is_Festival": 0,
    "Weather": "Storm", "Pickup_Zone": "CBD", "Dropoff_Zone": "Suburban",
    "Vehicle_Type": "Bike", "Rider_Experience_Years": 1.0, "Rider_Rating": 4.0,
    "Restaurant_Rating": 4.2, "Cuisine_Type": "Pizza", "Order_Items": 3,
    "Restaurant_Load": "High", "Preparation_Time_Min": 25, "Road_Distance_km": 9.2,
    "Traffic_Level": "High", "Number_of_Signals": 12, "Delivery_Priority": "Normal",
}

sample_df = pd.DataFrame([new_order])
sample_encoded = pd.get_dummies(sample_df).reindex(columns=model_columns, fill_value=0)

pred_minutes = rf_model.predict(sample_encoded)[0]


def format_minutes(x):
    h = int(x // 60)
    m = int(x % 60)
    return f"{h} hour(s) {m} minutes" if h > 0 else f"{m} minutes"

print(f"Predicted delivery time: {format_minutes(pred_minutes)}")


Predicted delivery time: 1 hour(s) 3 minutes
